# Reproduce everything

**Reconstruction-independent moment tests for quantum-computed dynamical spectra** — master notebook.

Every non-hardware result in the paper is an exact classical computation (`numpy`/`scipy`; a few
noise-model forecasts also use `qiskit-aer`). This notebook drives the scripts in `src/`, which write
the per-figure data fragments into `paper/figs/*.dat`. Run it from the repository root.

See [`docs/REPRODUCE.md`](../docs/REPRODUCE.md) for the figure → script map and the key numbers, and
[`docs/FILE_INDEX.md`](../docs/FILE_INDEX.md) for every file.

In [ ]:
import subprocess, sys, pathlib
ROOT = pathlib.Path.cwd()
assert (ROOT / "src" / "spectral_lanczos.py").exists(), "run this notebook from the repository root"

def run(script, *args, tail=2500):
    print(f"$ python src/{script} {' '.join(args)}".rstrip())
    r = subprocess.run([sys.executable, f"src/{script}", *args],
                       capture_output=True, text=True, cwd=ROOT)
    if r.stdout: print(r.stdout[-tail:])
    if r.returncode: print("‼ STDERR:", r.stderr[-1500:])
    print("-" * 70)
    return r.returncode

## 1 · Fast reproducibility check (seconds)

The current-probe moments, the shot-budget interval at the real `ibm_fez` budget, and the joint
`(m₀,m₁,m₂)`+Hankel battery catching a truncation the lone first moment misses.

In [ ]:
run("verify.py")

## 2 · The moment tests and the closed-form bounds (numpy/scipy)

The independent estimator, the interval-moment forecast and battery, the necessary∘sufficient
composition, the two-sided Markov–Krein bound, and the within-sector detectability analysis.

In [ ]:
for s in ["interval_moment.py", "interval_battery.py",
          "necessary_sufficient_composition.py", "markov_krein_window.py",
          "within_sector_control.py"]:
    run(s)

## 3 · The falsification demonstrations across observables and domains

Current-response Drude, collective channels, the negative-order (f-sum) test, the teeth
(independence vs a circular control), the shared-state leak, and the U(1) Gauss-law cross-domain
falsifier.

In [ ]:
for s in ["run_sumrule_falsifier.py", "run_collective_sumrule_falsifier.py",
          "run_inverse_moment_falsifier.py", "run_falsifier_teeth.py",
          "run_teeth_shared.py", "run_gausslaw_falsifier.py"]:
    run(s)

## 4 · The Christoffel / Gauss–Radau miss-distance certificate

In [ ]:
run("run_moment_bound_theorem.py")

## 5 · The blinded, pre-registered protocol (sealed)

The seal (`data/prereg.sha256`) is verified before generation; the frozen battery is then scored on
the sealed instances. Re-scoring reproduces the committed rates (truncation 56/56, spurious 45/61,
false-positive 1/112, Krylov 0/71).

In [ ]:
run("blind_score.py")

## 6 · Regenerate every native-figure data fragment

Each `export_*` writes its `paper/figs/*.dat`; the heatmap rasters are rebuilt by the `make_*` scripts.

In [ ]:
for s in ["export_akw_dat.py", "export_sqw_dat.py", "export_teeth_dat.py",
          "export_bracketing_dat.py", "export_christoffel_dat.py",
          "export_momentcone_dat.py"]:
    run(s)

## 7 · Noise-model forecasts and hardware (optional)

These use `qiskit-aer` and/or an IBM Quantum account and are **not** required to reproduce the paper's
figures (their outputs are committed under `data/`). Uncomment to run.

- `nk_falsifier.py` — the momentum-distribution `n_k` forecast (≈9 % of the Fermi step, FT-only).
- `offdiag_noise_forecast.py`, `offdiag_gsurface.py` — the off-diagonal depolarizing-bias floor (37–115 %).
- `interval_moment_mc.py` — the Monte-Carlo interval-moment test (`fig_momentmc`).
- `joint_covariance_composition.py`, `lever1_flip_experiment.py` — the same-sample (Durbin–Wu–Hausman) analysis.
- `hardware_matched_job_L8.py`, `run_heron_screen.py` — device-side jobs (**use your own IBM token; never commit it**).

In [ ]:
# run("nk_falsifier.py")
# run("interval_moment_mc.py")
# run("offdiag_noise_forecast.py")
print("optional cells — uncomment above to run")

## 8 · Build the paper

```bash
make paper          # -> paper/main.pdf  (uses the committed main.bbl; no bibtex)
```
Or upload `paper/arxiv-submission.tar.gz` directly (clean-room pdfLaTeX, 0 undefined references).